# មេរៀនទី ១០ - តួអង្គ AI ក្នុងការផលិត

ក្នុងមេរៀននេះ អ្នកនឹងរៀនពី **គំរូផលិតកម្ម** សម្រាប់តួអង្គ AI ដោយប្រើប្រាស់ Microsoft Agent Framework (Python)។ យើងគ្របដណ្តប់៖

- **ការសង្កេត** — ការបន្ថែមពេលវេលានិងការចុះផ្សាយទៅលើការទំនាក់ទំនងរបស់តួអង្គ
- **ការប៉ាន់ប្រមាណ** — ការប្រើប្រាស់តួអង្គវាយតម្លៃដើម្បីពិន្ទុគុណភាពរបាយការណ៍
- **ការគ្រប់គ្រងថ្លៃដើម** — វិធីសាស្រ្តសម្រាប់អុ្មតសញ្ញា និងការជ្រើសរើសម៉ូដែល

ស្ថានការណ៍គឺជាតួអង្គ **ការធ្វើដំណើរ** ដែលជួយអ្នកប្រើផែនការដំណើរផ្សងព្រេង ជាមួយនឹងការតាមដាន និងការវាយតម្លៃខ្នាតលើ។


## ការតំឡើង


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ការពិចារណារបស់ផលិតកម្ម

ការ​ដឹក​ជញ្ជូន​អ្នកAgent AI ពីគំរូ​ទៅ​រដ្ឋផលិតកម្ម​ត្រូវការ​ការយកចិត្តទុកដាក់យ៉ាងហ្មត់ចត់​ចំពោះជណ្តើរ​ទីបី៖

1. **ការមើលឃើញ (Observability)** — អ្នកត្រូវការមើលឃើញពីអ្វីដែលអ្នកAgent កំពុងធ្វើ រយៈពេលវាអាចយក និងឧបករណ៍ណាដែលវាអញ្ជើញ។ ឥតមានការតាមដាន និងកំណត់ត្រា ការពិនិត្យបញ្ហារដ្ឋផលិតកម្មគឺរឹងរាំងជាងរឿងកាយ។

2. **ការវាយតម្លៃ (Evaluation)** — ការត្រួតពិនិត្យគុណភាពដោយស្វ័យប្រវត្តិធានាថាចម្លើយរបស់អ្នកAgent នៅតែត្រឹមត្រូវ ពេញលេញ និងជួយគ្នាទៅវិញទៅមកកាលបរិច្ឆេទយូរ។ អ្នកAgent វាយតម្លៃអាចវាយពិន្ទុចម្លើយទាំងឡាយប៉ាន់ប្រមាណជាមួយកាតព្វកិច្ចបានកំណត់។

3. **ការគ្រប់គ្រងថ្លៃដើម (Cost Management)** — ការប្រើប្រាស់តួអក្សរមានឥទ្ឋិពលផ្ទាល់ទៅលើថ្លៃដើម។ យុទ្ធសាស្រ្តដូចជា ការបញ្ចុះបន្តិច បម្រែបម្រួលគំរូ និងការផ្ទុកក្នុងឃ្លាំងជួយរក្សាចំណាយឲ្យនៅក្រោមការត្រួតពិនិត្យដោយគ្មានការបាត់បង់គុណភាព។


## ការសង់តួនាទី Agent ដែលអាចមើលឃើញបាន

យើងកំណត់ឧបករណ៍ធ្វើដំណើរនិងវាស់ចំណុចប្រទាក់ agent ជាមួយការ៉េមកាលដើម្បីអាចតាមដានការពន្យារពេល។ នៅក្នុងផលិតកម្ម អ្នកនឹងបញ្ចូលជាមួយ OpenTelemetry ឬ backend តាមដានដូចគ្នា។


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## លំនាំវាយតម្លៃ

លំនាំផលិតជាទូទៅគឺការប្រើភ្នាក់ងារទីពីរជា **អ្នកវាយតម្លៃ** ។ អ្នកវាយតម្លៃវាយតម្លៃចម្លើយរបស់ភ្នាក់ងារមុខងារចម្បងប្រឆាំងនឹងលក្ខណៈដែលបានកំណត់ជាមុនដូចជា ភាពពេញលេញ, ភាពត្រឹមត្រូវ, និងភាពជួយជម្រះ។

នេះអនុញ្ញាតឲ្យមាន៖
- ទ្វារត qualitiy ដដែលស្វ័យប្រវត្តិមុនពេលចម្លើយឈានដល់អ្នកប្រើ
- ការស្គាល់ការធ្លាក់ចុះពេលបូកបន្ថែម ឬគំរូផ្លាស់ប្តូរ
- ការត្រួតពិនិត្យបន្តរអំពីសមត្ថភាពភ្នាក់ងារក្នុងរយៈពេលវែង


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## វិធីសាស្រ្តគ្រប់គ្រងថ្លៃដើម

ការត្រួតពិនិត្យថ្លៃដើម គឺមានសារៈសំខាន់សម្រាប់ភ្នាក់ងារបញ្ចេញ AI។ ទីនេះជាវិធីសាស្រ្តសំខាន់ៗ៖

| វិធីសាស្រ្ត | ការពិពណ៌នា |
|---|---|
| **បើកបរាបប្រើប្រាស់បញ្ជា** | រក្សារបញ្ជារបស់ប្រព័ន្ធឲ្យខ្លី។ ដកបរិបទដែលមិនចាំបាច់ចេញដើម្បីកាត់បន្ថយសញ្ញាចូល។ |
    "| **ជម្រើសម៉ូដែល** | ប្រើម៉ូដែលតូច និងថោកជាង (ឧ. GPT-4.1-mini) សម្រាប់ភារកិច្ចសាមញ្ញដូចជាការបែងចែក ឬការទាញយក ហើយរក្សាម៉ូដែលធំសម្រាប់គំនិតស្មុគស្មាញ។ |\n",
| **ការផ្ទុកទុក** | ផ្ទុកទុកលទ្ធផលឧបករណ៍ និងសំណួរញឹកញាប់ ដើម្បីជៀសវាងការហៅ API ម្តងទៀតតែម្តង។ |
| **ថវិកាសញ្ញា** | កំណត់ `max_tokens` ដើម្បីបង្ការជម្លុះជម្រះចម្លើយវែងពេក។ |
| **ការបញ្ចូលជាក្រុម** | ក្រុមការសួររបស់អ្នកប្រើជាច្រើនទៅក្នុងការហៅ API មួយដដែលនៅពេលដែលអាចធ្វើបាន។ |

ក្នុងការអនុវត្តន៍ វិធីសាស្រ្តជាបន្ទាន់កម្រិតធ្វើអោយមានប្រសិទ្ធភាព៖ ម្ចាស់ការសំណើងាយៗទៅម៉ូដែលដែលរហ័ស និងថោក ហើយលើកកម្ពស់តែសំណួរលំបាកទៅម៉ូដែលដែលមានសមត្ថភាពខ្ពស់។ 


## សេចក្ដីសង្ខេប

ក្នុងមេរៀននេះ អ្នកបានរៀនពីវិធីខាងក្រោម៖

1. **បន្ថែមការត្រួតពិនិត្យ** ចំពោះប្រតិបត្តិការ​អេហ្សង់ដោយប្រើពេលវេលា និងកំណត់ហេតុ ដើម្បីដាក់មូលដ្ឋានសម្រាប់ការតាមដាន និងត្រួតពិនិត្យ។
2. **វាយតម្លៃចម្លើយអេហ្សង់** អោយបានអូតូម៉ាទិច ដោយប្រើអេហ្សង់វាយតម្លៃដែលវាស់វែងភាពសព្វវរគ្រប់គ្រាន់ ភាពត្រឹមត្រូវ និងភាពមានប្រយោជន៍។
3. **គ្រប់គ្រងថ្លៃដើម** តាមរយៈការអុបផ្ទេមរឺលើកទាញការបញ្ចូល ប្តូរគំរូ ការផ្ទុកត្រឡប់ និងថវិកាតូកិន។

លំនាំផលិតកម្មទាំងនេះជួយធានាថា អេហ្សង់ AI របស់អ្នកមានភាពទុកចិត្ត បានវាស់វែង និងមានប្រសិទ្ធិភាពថ្លៃដើមនៅក្នុងការប្រើប្រាស់វិសាល។ 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
